# Gradiente Descencente para polinomio de grau 17
O presente código demonstra a implementação de um algoritmo com o intuito de obter os coeficientes de uma função polinomial.

## Imports

In [1]:
import tensorflow as tf
import random

2024-11-01 02:35:05.479848: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Funções

`gerar_sequencia` é uma função de apoio para criação do dataset de entrada.

In [2]:
def gerar_sequencia(num_elementos, valor_inicial, valor_final):
    return [valor_inicial + (valor_final - valor_inicial) * i / (num_elementos - 1) for i in range(num_elementos)]

A função `f` representa a função polinomial f(x) de grau a-1

In [3]:
def f(a,x):
    n_a = a.shape[0]
    return sum(a[j] * (x ** ((n_a-1) - j)) for j in range(n_a))

`l_loss` função loss.
$$\begin{align}
&(y_1-(a_1x_1^{17}+a_2x_1^{16}+a_3x_1^{15}+a_4x_1^{14}+a_5x_1^{13}+a_6x_1^{12}+a_7x_1^{11}\\
&+a_8x_1^{10}+a_9x_1^{9}+a_{10}x_1^{8}+a_{11}x_1^{7}+a_{12}x_1^{6}+a_{13}x_1^{5}+a_{14}x_1^{4}\\
&+a_{15}x_1^{3}+a_{16}x_1^{2}+a_{17}x_1+a_{18}))^2\\
&+\cdots+\\
&(y_n-(a_1x_n^{17}+a_2x_n^{16}+a_3x_n^{15}+a_4x_n^{14}+a_5x_n^{13}+a_6x_n^{12}+a_7x_n^{11}+a_8x_n^{10}\\
&+a_9x_n^{9}+a_{10}x_n^{8}+a_{11}x_n^{7}+a_{12}x_n^{6}+a_{13}x_n^{5}+a_{14}x_n^{4}+a_{15}x_n^{3}\\
&+a_{16}x_n^{2}+a_{17}xn+a_{18}))^2
\end{align}$$

In [4]:
def f_loss(a, x, y):
    n_a = a.shape[0]
    l = 0.0
    for i in range(y.shape[0]):
        polinomial = f(a,x[i])
        l += (y[i] - polinomial) ** 2
    return l

In [5]:
def grad(a,x,y):
    with tf.GradientTape() as tape:
        loss = f_loss(a, x, y)

    # Compute the gradients of the loss with respect to a
    grads = tape.gradient(loss, a)
    return grads

In [37]:
def dist(a_n, a_o):
    return tf.norm(a_n - a_o).numpy()

In [41]:
def grad_desc(grau, dataset_x, dataset_y,tol,lr):
    coefs=grau+1
    x = tf.constant(dataset_x)  # Example input values
    y = tf.constant(dataset_y)
    
    # coeficientes iniciais aleatórios
    a_o = tf.Variable(tf.random.normal([coefs]), name='coefs_init')

    a_n = tf.Variable(a_o.numpy()) # iniciais
    i = 0
    
    while True:
        gradients = grad(a_n, x, y)  # Get gradients for all coefficients
        a_n.assign(a_n - lr * gradients)  # Update all coefficients
        
        i += 1
        if dist(a_n, a_o) > tol:
            a_o.assign(a_n)  # Update the old coefficients to the new ones
        else:
            break
    return a_o.numpy(), i

In [42]:
grau = 17
coefs = grau + 1

# Target randomico
#coefs_target = [round(random.random(), 2) for _ in range(coefs)]
coefs_target = [random.uniform(-3, 3) for _ in range(coefs)]

print("coeficientes esperados:",coefs_target)

# Gerando data_set com base nos coefs_target
a = tf.Variable(coefs_target, name='coefs')

dataset_x = gerar_sequencia(30,-2,2)
dataset_y = [f(a, x).numpy() for x in dataset_x]


coeficientes esperados: [-0.935863212773075, -0.6949806583801115, 1.2521641954267873, -1.4171013471604677, -2.2963438478393043, -2.052148006833881, -1.0074699517057215, 2.3430853263677394, -0.43854600029277613, 2.2706411485689317, -1.2282083943528201, 1.269734237813033, -2.689132974248217, -2.5661042075831615, 1.2067659221963574, 2.7703411325280207, -1.4048123882106924, 1.4978768810739753]


In [43]:
gd, i = grad_desc(grau, dataset_x, dataset_y,10**(-16),0.01)
print(gd,i)

[7.6724806e+35 8.1370487e+32 1.9464706e+35 2.0694396e+32 4.9503563e+34
 5.2787345e+31 1.2627152e+34 1.3512363e+31 3.2322577e+33 3.4734615e+30
 8.3090497e+32 8.9755602e+29 2.1470283e+32 2.3338883e+29 5.5830931e+31
 6.1172803e+28 1.4632949e+31 1.6193029e+28] 5


In [19]:
print(dataset_x[0], dataset_y[0])

-2.0 -17025.656


In [20]:
print(f(gd, dataset_x[0]))

-4.995022524582396e+36
